# WSJCAM0 Autoregressive Speech Prediction

This notebook bootstraps the environment, trains FACodec and EnCodec speech predictors, evaluates them with STOI/PESQ/DNSMOS, and exports comparison artifacts.

Important:

1. Run the bootstrap cells first.
2. Switch the kernel once to the newly registered `finalproject26-py39` kernel.
3. Run the remaining cells with `Run All`.


In [ ]:
from pathlib import Path
import os

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
os.chdir(ROOT)
print(f'Project root: {ROOT}')
print(f'Dataset root exists: {(ROOT / "wsj0").exists()}')


In [ ]:
import shlex
import subprocess

ENV_NAME = 'finalproject26-py39'
PYTHON_VERSION = '3.9'
TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
TORCH_VERSION = '2.8.0'
TORCHVISION_VERSION = '0.23.0'
TORCHAUDIO_VERSION = '2.8.0'

def bash(command: str) -> None:
    print(command)
    subprocess.run(['bash', '-lc', command], check=True)


In [ ]:
requirements_path = shlex.quote(str((ROOT / 'requirements.txt').resolve()))

bash(
    f"if ! conda env list | awk '{'{print $1}'}' | grep -qx {shlex.quote(ENV_NAME)}; then "
    f"conda create -y -n {shlex.quote(ENV_NAME)} python={PYTHON_VERSION}; fi"
)
bash(f"conda run -n {shlex.quote(ENV_NAME)} python -m pip install --upgrade pip setuptools wheel")
bash(
    f"conda run -n {shlex.quote(ENV_NAME)} pip install "
    f"torch=={TORCH_VERSION} torchvision=={TORCHVISION_VERSION} torchaudio=={TORCHAUDIO_VERSION} "
    f"--index-url {TORCH_INDEX_URL}"
)
bash(f"conda run -n {shlex.quote(ENV_NAME)} pip install -r {requirements_path}")
bash(
    f"conda run -n {shlex.quote(ENV_NAME)} python -m ipykernel install --user "
    f"--name {shlex.quote(ENV_NAME)} --display-name 'Python ({ENV_NAME})'"
)
print('\nBootstrap finished. Switch the notebook kernel to Python (finalproject26-py39) before continuing.')


## Stop Here Once

After the previous cell succeeds:

- open the kernel selector
- switch to `Python (finalproject26-py39)`
- continue with the cells below


In [ ]:
import json
import sys

import pandas as pd
import torch
import torchaudio

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Torchaudio:', torchaudio.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from src.data.manifest import build_wsj0_manifest

train_manifest = build_wsj0_manifest('wsj0', 'si_tr_s')
val_manifest = build_wsj0_manifest('wsj0', 'si_dt_05')
test_manifest = build_wsj0_manifest('wsj0', 'si_et_05')
pd.DataFrame([
    {'split': 'si_tr_s', 'num_files': len(train_manifest)},
    {'split': 'si_dt_05', 'num_files': len(val_manifest)},
    {'split': 'si_et_05', 'num_files': len(test_manifest)},
])


In [ ]:
from src.experiment import compare_codecs, run_codec_experiment

SMOKE_RUN = False
DATASET_ROOT = 'wsj0'
ARTIFACTS_DIR = 'artifacts'

COMMON_OVERRIDES = {
    'dataset': {
        'root': DATASET_ROOT,
    },
    'project': {
        'artifacts_dir': ARTIFACTS_DIR,
    },
}

if SMOKE_RUN:
    COMMON_OVERRIDES = {
        **COMMON_OVERRIDES,
        'training': {
            'batch_size': 2,
            'num_workers': 0,
            'max_epochs': 1,
            'context_frames': 40,
        },
        'evaluation': {
            'max_eval_files': 8,
            'save_audio_examples': 2,
        },
    }

COMMON_OVERRIDES


In [ ]:
facodec_config, facodec_train, facodec_eval = run_codec_experiment('facodec', overrides=COMMON_OVERRIDES)
print(facodec_train)
print(facodec_eval)


In [ ]:
encodec_config, encodec_train, encodec_eval = run_codec_experiment('encodec', overrides=COMMON_OVERRIDES)
print(encodec_train)
print(encodec_eval)


In [ ]:
summary_table = compare_codecs({
    'facodec': facodec_eval.summary_path,
    'encodec': encodec_eval.summary_path,
})
summary_table.to_csv(Path(ARTIFACTS_DIR) / 'codec_comparison.csv', index=False)
summary_table
